# Class 10 activity — Look at the data before you build the server

**~15 minutes, then B4 on your own after class.** Then we close this notebook and go build the API in a
terminal, from files.

That order is deliberate. Lab 5 asks you to put an HTTP server in front of a
database. It is very hard to tell whether a server is returning the *right*
answer if you have never seen the data by hand. So: SQL first, by hand, with no
Python in the way. Then the same query through an ORM, watching what it writes
for you.

Notebooks are good for exactly this — poking at data, seeing tables, changing
one thing and re-running. They are not good for servers, which is why your
actual Lab 5 will be `.py` files and a terminal.

---
## Part A — SQL, before any Python

Run this cell. It builds a small seeded database and connects a SQL engine to it.

In [3]:
# This notebook needs SQLModel and jupysql. They're there if JupyterLab was
# started with `uv run jupyter lab` in the folder where you ran `uv add`.
try:
    import sqlmodel, sql  # noqa: F401
except ImportError:
    raise SystemExit(
        "Missing packages. Close JupyterLab, then in lecture-activities/lecture-10 run "
        "`uv add sqlmodel jupysql pandas jupyterlab` and `uv run jupyter lab`. "
        "(See 'Start here' at the top of the build-along.)"
    )

import sqlite3, pathlib
pathlib.Path("practice.db").unlink(missing_ok=True)
con = sqlite3.connect("practice.db")
con.executescript('''
CREATE TABLE idea (id INTEGER PRIMARY KEY, title TEXT, author TEXT);
CREATE TABLE vote (id INTEGER PRIMARY KEY, idea_id INTEGER, voter TEXT);
INSERT INTO idea (title, author) VALUES
  ('Automatic plant waterer','ada'),
  ('Laundry that folds itself','grace'),
  ('Bike lock that texts you','alan'),
  ('A better course scheduler','ada');
INSERT INTO vote (idea_id, voter) VALUES (1,'grace'),(1,'alan'),(2,'ada'),(1,'ada'),(4,'grace');
''')
con.commit(); con.close()

import warnings  # jupysql emits a harmless SyntaxWarning on import
warnings.filterwarnings("ignore", category=SyntaxWarning)
%load_ext sql
%sql sqlite:///practice.db
print("practice.db ready")

/workspaces/E190BO_Lab_Repo/lecture-activities/lecture-10/.venv/lib/python3.13/site-packages/sql/parse.py:339: SyntaxWarning: invalid escape sequence '\:'
  Given a query, replaces all occurrences of ':variable' with '\:variable' and
/workspaces/E190BO_Lab_Repo/lecture-activities/lecture-10/.venv/lib/python3.13/site-packages/sql/parse.py:369: SyntaxWarning: invalid escape sequence '\:'
  Given a query, replaces all occurrences of 'example'[x:y] with 'example'[x\:y].


The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'sqlite:///practice.db'

practice.db ready


A cell starting with `%%sql` is **SQL, not Python** — no quotes, no string
escaping, results come back as a table:

In [4]:
%%sql
SELECT * FROM idea;

Running query in 'sqlite:///practice.db'

id,title,author
1,Automatic plant waterer,ada
2,Laundry that folds itself,grace
3,Bike lock that texts you,alan
4,A better course scheduler,ada


**A1.** Every idea written by `ada`:

In [11]:
%%sql
SELECT * FROM idea WHERE author == "ada" ;

Running query in 'sqlite:///practice.db'

id,title,author
1,Automatic plant waterer,ada
4,A better course scheduler,ada


**A2.** Each idea with **how many votes it has**, most-voted first.

You will need `JOIN`, `GROUP BY`, `COUNT` and `ORDER BY`. Take your time — this
is the query behind Lab 5's `GET /ideas`, which returns every idea with its
`vote_count`. It is much easier to get right here, where it is only SQL.

Watch out for *Bike lock that texts you*. It has no votes. Does it appear in
your results? Should it?

In [ ]:
%%sql


**A3.** Why does A2 need a `JOIN` at all? What goes wrong if you query only `vote`?

> **✏️ Your answer:**
>
> *Double-click this cell, replace this line with your answer, then press Shift+Enter.*

---
## Part B — The same queries, through an ORM

In Lab 5 you will not write that SQL. You will write Python classes, and an
**ORM** (Object-Relational Mapper) will write the SQL for you.

That is a good deal, right up until the moment the SQL it writes is not what you
expected. So today we watch it work. `echo=True` prints every statement the ORM
sends to the database — and at the Lab 5 check-in you'll be asked to read yours.

In [1]:
from sqlmodel import SQLModel, Field, Session, create_engine, select, func

class Idea(SQLModel, table=True):
    __tablename__ = "idea"
    id: int | None = Field(default=None, primary_key=True)
    title: str
    author: str = ""

class Vote(SQLModel, table=True):
    __tablename__ = "vote"
    id: int | None = Field(default=None, primary_key=True)
    idea_id: int = Field(foreign_key="idea.id")
    voter: str

# echo=True is the whole point of this cell
engine = create_engine("sqlite:///practice.db", echo=True)

**B1.** Your A1 again, through the ORM: every idea by `ada`, printed by title.

Use a `Session`, `select(Idea)`, and `.where(Idea.author == "ada")`. Then look at
what got printed *above* your output — that is the SQL the ORM wrote.

In [2]:
with Session(engine) as sess:
    pass  # 

**B2.** Compare the SQL the ORM emitted with what you wrote in A1.
Name **two** differences. Where did `"ada"` go?

> **✏️ Your answer:**
>
> *Double-click this cell, replace this line with your answer, then press Shift+Enter.*

**B3.** Now A2 — the one that matters for Lab 5. Here it is through the ORM,
with one thing wrong. **Run it as-is**, then compare the output with your A2.

In [ ]:
stmt = (
    select(Idea.title, func.count(Vote.id))
    .join(Vote)
    .group_by(Idea.id)
    .order_by(func.count(Vote.id).desc())
)
with Session(engine) as sess:
    for title, n in sess.exec(stmt):
        print(n, title)

Which idea is missing, and why? Fix it in the cell below: copy the statement and
change **one** thing. (Look up `.join` in the SQLModel/SQLAlchemy docs, or read
the SQL that got printed and ask which word your A2 had that this doesn't.)

In [ ]:
# YOUR FIX HERE

> **✏️ Your answer:**
>
> *Double-click this cell, replace this line with your answer, then press Shift+Enter.*

**B4 (after class, 5 minutes).** Here is another way to get the same numbers, and it
is the one most people (and most AI assistants) write first. Run it, then
**count the `SELECT` statements** in the log.

In [ ]:
with Session(engine) as sess:
    for idea in sess.exec(select(Idea)).all():
        votes = sess.exec(select(Vote).where(Vote.idea_id == idea.id)).all()
        print(len(votes), idea.title)

How many `SELECT`s was that? How many would it be with 1,000 ideas? How many did B3 take?

> **✏️ Your answer:**
>
> *Double-click this cell, replace this line with your answer, then press Shift+Enter.*

> **✅ Check yourself** — these are the kinds of questions you'll be asked at the Lab 5 check-in. Can you answer them out loud, in your own words?
>
> - Why is this data in two tables instead of one?
> - What is the difference between a `JOIN` and a `LEFT JOIN` here? Which idea tells them apart?
> - Your ORM wrote SQL for you. What did it write for B3, and why is there a `?` in B1's?
> - How many queries does your way of counting votes send, for 4 ideas? For 1,000?

---
## Next: close this notebook

The rest of class is in a terminal, in files, in your own repo. You'll start the
API that serves this data over HTTP — and Lab 5 is where you finish it.

Your Lab 5 database has these same two tables, and `GET /ideas` has to return
B3's answer as JSON: every idea, with a `vote_count`.